# Project Setup

Initialize imports, reproducibility settings, and project paths.

## Starter checklist
- Load core scientific Python libraries.
- Set random seeds for reproducibility.
- Create output directories if needed.
- Confirm the workspace is ready for notebook-based analysis.


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path(r'C:/Bioinformatics Project')
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
RESULTS_FIGURES = ROOT / 'results' / 'figures'
RESULTS_TABLES = ROOT / 'results' / 'tables'
for path in [DATA_RAW, DATA_PROCESSED, RESULTS_FIGURES, RESULTS_TABLES]:
    path.mkdir(parents=True, exist_ok=True)

print("Notebook environment ready.")
print(f"ROOT={ROOT}")
print(f"DATA_RAW={DATA_RAW}")
print(f"DATA_PROCESSED={DATA_PROCESSED}")
print(f"RESULTS_FIGURES={RESULTS_FIGURES}")
print(f"RESULTS_TABLES={RESULTS_TABLES}")


## Dataset path and profile check

This cell verifies that the GEO raw folders exist and summarizes the transcriptomic profile stored in each dataset.


In [ ]:
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(r'C:/Bioinformatics Project')
DATASET_RAW = PROJECT_ROOT / 'dataset' / 'raw'
ALT_DATA_RAW = PROJECT_ROOT / 'data' / 'raw'

DATASETS = {
    'GSE26168': {
        'folder': DATASET_RAW / 'GSE26168_RAW',
        'profile': 'Human whole-blood microarray transcriptomic profile (Affymetrix/Illumina probe-level data)',
        'expected_patterns': ['*.bgx', '*.CEL', '*.CEL.gz', '*.bgx.gz'],
    },
    'GSE15932': {
        'folder': DATASET_RAW / 'GSE15932_RAW',
        'profile': 'Human peripheral blood mononuclear cell (PBMC) microarray transcriptomic profile',
        'expected_patterns': ['*.CEL', '*.CEL.gz', '*.CHP', '*.CHP.gz'],
    },
    'GSE164416': {
        'folder': DATASET_RAW / 'GSE164416_RAW',
        'profile': 'Human pancreatic islet RNA-seq transcriptomic profile with HTSeq count files',
        'expected_patterns': ['*.txt.gz', '*.txt'],
    },
}

def check_dataset_folder(dataset_name: str, dataset_info: dict) -> dict:
    folder = dataset_info['folder']
    if not folder.exists() and (ALT_DATA_RAW / folder.name).exists():
        folder = ALT_DATA_RAW / folder.name

    dataset_info['resolved_folder'] = folder
    exists = folder.exists()
    file_count = sum(1 for item in folder.rglob('*') if item.is_file()) if exists else 0
    matches = []
    if exists:
        for pattern in dataset_info['expected_patterns']:
            matches.extend(folder.rglob(pattern))

    if matches:
        sample_files = sorted({path.name for path in matches})[:5]
        evidence_files = ', '.join(sample_files)
    else:
        evidence_files = 'none found for expected file patterns'

    status = 'READY' if exists and file_count > 0 else 'MISSING / EMPTY'
    return {
        'Dataset': dataset_name,
        'Folder': str(folder.relative_to(PROJECT_ROOT)) if folder.is_relative_to(PROJECT_ROOT) else str(folder),
        'Exists': exists,
        'Files': file_count,
        'Transcriptomic profile': dataset_info['profile'],
        'Evidence files': evidence_files,
        'Status': status,
    }

summary_rows = [check_dataset_folder(name, info) for name, info in DATASETS.items()]
summary_df = pd.DataFrame(summary_rows)[['Dataset', 'Status', 'Files', 'Transcriptomic profile', 'Evidence files', 'Folder']]
display(summary_df.style.hide(axis='index'))

all_ready = all(row['Status'] == 'READY' for row in summary_rows)
print('All dataset folders are ready.' if all_ready else 'One or more dataset folders are missing or empty.')
